In [ ]:
import pandas as pd
import numpy as np
from sksurv.util import Surv
from sksurv.ensemble import RandomSurvivalForest
from sksurv.metrics import concordance_index_ipcw
from sklearn.model_selection import train_test_split

In [ ]:
train_df = pd.read_csv("imputed_train.csv")
test_df = pd.read_csv("imputed_test.csv")

In [ ]:
#convert everything to numeric for LASSO
train_df = train_df.apply(pd.to_numeric)
test_df = test_df.apply(pd.to_numeric)

In [ ]:
train_df["GSTATUS"].sum() + test_df["GSTATUS"].sum()

In [ ]:
y_train = Surv.from_dataframe("GSTATUS", "GTIME", train_df)
y_test = Surv.from_dataframe("GSTATUS", "GTIME", test_df)

X_train = train_df.drop(columns=["GSTATUS", "GTIME"], axis=1)
X_test = test_df.drop(columns=["GSTATUS", "GTIME"], axis=1)

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import (
    concordance_index_censored,
    concordance_index_ipcw,
    cumulative_dynamic_auc,
    brier_score,
    integrated_brier_score,
)
from sksurv.util import Surv


# ============================================================
# 0) INPUTS EXPECTED
# ------------------------------------------------------------
# X_train, X_test : pandas DataFrame or numpy array
# y_train, y_test : structured survival arrays OR DataFrames with
#                   columns ["GSTATUS", "GTIME"]
#                   where GSTATUS is event indicator and GTIME is time
# ============================================================


# ----------------------------
# Survival-object helpers
# ----------------------------
def ensure_surv(y):
    """
    Return sksurv Surv structured array with fields:
    event = GSTATUS (bool), time = GTIME (float)
    """
    if isinstance(y, np.ndarray) and y.dtype.names is not None:
        # already structured
        return y

    if isinstance(y, pd.DataFrame):
        yy = y.copy()
    else:
        yy = pd.DataFrame(y, columns=["GSTATUS", "GTIME"])

    yy["GSTATUS"] = yy["GSTATUS"].astype(bool)
    yy["GTIME"] = yy["GTIME"].astype(float)
    return Surv.from_dataframe("GSTATUS", "GTIME", yy)


def harrell_c(y_true, risk_score):
    return concordance_index_censored(
        y_true["GSTATUS"], y_true["GTIME"], risk_score
    )[0]


def ipcw_c(y_train, y_test, risk_score, tau=None):
    """
    IPCW concordance. If tau is None, choose a safe tau slightly below
    the max follow-up in the test set, but not beyond train support.
    """
    train_max = float(np.max(y_train["GTIME"]))
    test_max = float(np.max(y_test["GTIME"]))

    if tau is None:
        tau = min(train_max, test_max) - 1e-8

    return concordance_index_ipcw(y_train, y_test, risk_score, tau=tau)[0]


def get_supported_times(y_train, desired_times):
    """
    Keep only times strictly within training follow-up support.
    """
    desired_times = np.asarray(desired_times, dtype=float)
    train_max = float(np.max(y_train["GTIME"]))
    supported = desired_times[desired_times < train_max]

    if supported.size == 0:
        raise ValueError(
            f"No requested times are supported. Max train follow-up = {train_max:.1f}"
        )
    return supported


# ----------------------------
# Cross-validated alpha selection
# ----------------------------
def select_alpha_cv(
    X_train,
    y_train,
    l1_ratio,
    n_splits=5,
    n_alphas=100,
    alpha_min_ratio="auto",
    max_iter=20000,
    random_state=42,
    ridge_eps=1e-6,
):
    """
    Select alpha by CV using Harrell's C-index.

    Strategy:
    1) Fit once on full training data to get a common alpha grid.
    2) Refit inside each CV fold using that same grid.
    3) Score every alpha on the validation fold.
    4) Pick alpha with best mean CV C-index.
    """
    X_train = X_train.copy()
    y_train = ensure_surv(y_train)

    # scikit-survival ridge is usually approximated with tiny l1_ratio
    if l1_ratio == 0:
        l1_ratio = ridge_eps

    # scale once only to derive a stable global alpha grid
    scaler0 = StandardScaler()
    X0 = scaler0.fit_transform(X_train)

    probe = CoxnetSurvivalAnalysis(
        l1_ratio=l1_ratio,
        n_alphas=n_alphas,
        alpha_min_ratio=alpha_min_ratio,
        max_iter=max_iter,
        fit_baseline_model=False,
    )
    probe.fit(X0, y_train)
    alpha_grid = probe.alphas_

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_scores = []

    for tr_idx, va_idx in cv.split(X_train):
        X_tr = X_train.iloc[tr_idx] if hasattr(X_train, "iloc") else X_train[tr_idx]
        X_va = X_train.iloc[va_idx] if hasattr(X_train, "iloc") else X_train[va_idx]
        y_tr = y_train[tr_idx]
        y_va = y_train[va_idx]

        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr)
        X_va_s = scaler.transform(X_va)

        model = CoxnetSurvivalAnalysis(
            l1_ratio=l1_ratio,
            alphas=alpha_grid,              # force same grid in every fold
            max_iter=max_iter,
            fit_baseline_model=False,
        )
        model.fit(X_tr_s, y_tr)

        scores_this_fold = []
        for a in alpha_grid:
            risk_va = model.predict(X_va_s, alpha=a)
            scores_this_fold.append(harrell_c(y_va, risk_va))

        fold_scores.append(scores_this_fold)

    fold_scores = np.asarray(fold_scores)              # shape: (n_folds, n_alphas)
    mean_scores = fold_scores.mean(axis=0)
    best_idx = int(np.argmax(mean_scores))

    return {
        "alpha_grid": alpha_grid,
        "cv_scores": fold_scores,
        "mean_cv_scores": mean_scores,
        "best_alpha": float(alpha_grid[best_idx]),
        "best_cv_cindex": float(mean_scores[best_idx]),
    }


# ----------------------------
# Final fit + evaluation
# ----------------------------
def fit_and_evaluate_coxnet(
    model_name,
    X_train,
    y_train,
    X_test,
    y_test,
    l1_ratio,
    n_splits_cv=5,
    n_alphas=100,
    alpha_min_ratio="auto",
    max_iter=20000,
    auc_times=(365.0, 730.0),
    ibs_times=None,
    random_state=42,
):
    """
    Fit penalized Cox model with CV-selected alpha, then evaluate on test set.
    """
    y_train = ensure_surv(y_train)
    y_test = ensure_surv(y_test)

    # choose alpha
    cv_info = select_alpha_cv(
        X_train=X_train,
        y_train=y_train,
        l1_ratio=l1_ratio,
        n_splits=n_splits_cv,
        n_alphas=n_alphas,
        alpha_min_ratio=alpha_min_ratio,
        max_iter=max_iter,
        random_state=random_state,
    )
    best_alpha = cv_info["best_alpha"]

    # final fit on full training set
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    final_model = CoxnetSurvivalAnalysis(
        l1_ratio=(1e-6 if l1_ratio == 0 else l1_ratio),
        alphas=cv_info["alpha_grid"],      # same alpha grid as CV
        max_iter=max_iter,
        fit_baseline_model=True,           # needed for survival functions
    )
    final_model.fit(X_train_s, y_train)

    # risk scores at selected alpha
    risk_test = final_model.predict(X_test_s, alpha=best_alpha)

    # supported horizons
    auc_times = get_supported_times(y_train, auc_times)

    # AUC from risk scores
    auc_values, mean_auc = cumulative_dynamic_auc(
        y_train, y_test, risk_test, auc_times
    )

    # survival probabilities for Brier / IBS
    surv_funcs = final_model.predict_survival_function(
        X_test_s, alpha=best_alpha, return_array=False
    )
    S_test_auc = np.vstack([fn(auc_times) for fn in surv_funcs])

    _, brier_values = brier_score(y_train, y_test, S_test_auc, auc_times)

    # IBS grid
    if ibs_times is None:
        ibs_times = np.linspace(30.0, min(730.0, np.max(y_train["GTIME"]) - 1e-8), 25)
    else:
        ibs_times = get_supported_times(y_train, ibs_times)

    S_test_ibs = np.vstack([fn(ibs_times) for fn in surv_funcs])
    ibs = integrated_brier_score(y_train, y_test, S_test_ibs, ibs_times)

    result = {
        "model": model_name,
        "best_alpha": best_alpha,
        "best_cv_cindex": cv_info["best_cv_cindex"],
        "harrell_c": harrell_c(y_test, risk_test),
        "ipcw_c": ipcw_c(y_train, y_test, risk_test),
        "auc_times": auc_times,
        "auc_values": auc_values,
        "mean_auc": float(mean_auc),
        "brier_times": auc_times,
        "brier_values": brier_values,
        "ibs": float(ibs),
        "n_nonzero": int(np.sum(final_model.coef_[:, np.argmin(np.abs(final_model.alphas_ - best_alpha))] != 0)),
        "model_obj": final_model,
        "scaler": scaler,
        "cv_info": cv_info,
    }
    return result


# ----------------------------
# Run models
# ----------------------------
ridge_res = fit_and_evaluate_coxnet(
    model_name="Cox (Ridge-like)",
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    l1_ratio=0.0,                 # internally converted to tiny epsilon
    n_splits_cv=5,
    n_alphas=100,
    alpha_min_ratio="auto",
    max_iter=20000,
    auc_times=(365.0, 730.0),
    random_state=42,
)

enet_res = fit_and_evaluate_coxnet(
    model_name="Cox (Elastic Net)",
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    l1_ratio=0.5,                 # change if you want 0.1 / 0.25 / 0.75
    n_splits_cv=5,
    n_alphas=100,
    alpha_min_ratio="auto",
    max_iter=20000,
    auc_times=(365.0, 730.0),
    random_state=42,
)

# Optional lasso if you still want to try it
lasso_res = fit_and_evaluate_coxnet(
    model_name="Cox (LASSO)",
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    l1_ratio=1.0,
    n_splits_cv=5,
    n_alphas=100,
    alpha_min_ratio="auto",
    max_iter=50000,
    auc_times=(365.0, 730.0),
    random_state=42,
)


# ----------------------------
# Pretty table
# ----------------------------
def summarize_result(res):
    out = {
        "Model": res["model"],
        "Best alpha": res["best_alpha"],
        "Best CV C": res["best_cv_cindex"],
        "Harrell C": res["harrell_c"],
        "IPCW C": res["ipcw_c"],
        "Mean AUC": res["mean_auc"],
        "IBS": res["ibs"],
        "Non-zero coeffs": res["n_nonzero"],
    }
    for t, a in zip(res["auc_times"], res["auc_values"]):
        out[f"AUC@{int(t)}d"] = a
    for t, b in zip(res["brier_times"], res["brier_values"]):
        out[f"Brier@{int(t)}d"] = b
    return out

df_results = pd.DataFrame([
    summarize_result(ridge_res),
    summarize_result(enet_res),
    summarize_result(lasso_res),
])

print(df_results)

In [ ]:
import optuna
import numpy as np
from sklearn.model_selection import StratifiedShuffleSplit
from sksurv.metrics import concordance_index_censored

def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 50),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 50),
        "max_features": trial.suggest_float("max_features", 0.2, 1.0),
    }

    splitter = StratifiedShuffleSplit(n_splits=2, test_size=0.25, random_state=42)
    event = y_train["GSTATUS"].astype(int)

    scores = []
    for tr_idx, va_idx in splitter.split(X_train, event):
        m = RandomSurvivalForest(**params, random_state=trial.number, n_jobs=-1)   # your RSF constructor
        m.fit(X_train.iloc[tr_idx], y_train[tr_idx])
        risk = m.predict(X_train.iloc[va_idx])
        c = concordance_index_censored(y_train[va_idx]["GSTATUS"], y_train[va_idx]["GTIME"], risk)[0]
        scores.append(c)

    return float(np.mean(scores))

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40)

best_params = study.best_params


In [ ]:
#n_estimators': 304, 'max_depth': 5, 'min_samples_split': 46, 'min_samples_leaf': 11, 'max_features': 0.46199700080828915
rsf = RandomSurvivalForest(n_estimators=304, max_depth=5, min_samples_leaf=11, min_samples_split=46, max_features= 0.46199700080828915,
                           bootstrap=True, verbose=1, n_jobs=-1, random_state=42)

In [ ]:
rsf.fit(X_train, y_train)

In [ ]:
rsf.score(X_train, y_train)

In [ ]:
import matplotlib.pyplot as plt
from sksurv.metrics import cumulative_dynamic_auc



y_train = pd.DataFrame(y_train, columns=["GSTATUS", "GTIME"])
y_test = pd.DataFrame(y_test, columns=["GSTATUS", "GTIME"])

y_train["GSTATUS"] = y_train["GSTATUS"].astype("bool")
y_test["GSTATUS"] = y_test["GSTATUS"].astype("bool")

# Prepare survival objects for train and test data
y_test_surv = Surv.from_dataframe("GSTATUS", "GTIME", data=y_test)
y_train_surv = Surv.from_dataframe("GSTATUS", "GTIME", data=y_train)


times = np.arange(365, 365*7+1, 365)

rsf_scores = rsf.predict_cumulative_hazard_function(X_test, return_array=False)
rsf_surv = np.vstack([chf(times) for chf in rsf_scores])

rsf_auc, rsf_mean_auc = cumulative_dynamic_auc(y_train_surv, y_test_surv, rsf_surv, times)


plt.plot(times, rsf_auc, "o-", label=f"RSF (mean AUC = {rsf_mean_auc:.3f})")
plt.xlabel("days from transplant")
plt.ylabel("time-dependent AUC")
plt.legend(loc="lower center")
plt.grid(True)




In [ ]:
from sksurv.metrics import concordance_index_ipcw

# y_train_surv and y_test_surv are your Surv objects
rsf_risk_test = rsf.predict(X_test)

tau = min(y_train_surv["GTIME"].max(), y_test_surv["GTIME"].max()) - 1e-8
# tau = 365*2+1
rsf_ipcw_c = concordance_index_ipcw(
    y_train_surv, y_test_surv, rsf_risk_test, tau=tau
)[0]

print("RSF IPCW C-index:", rsf_ipcw_c)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter

def calibration_at_horizon(surv_funcs, y_time, y_event, horizon=365, n_bins=5):
    # predicted risk by horizon
    pred_risk = 1 - np.array([fn(horizon) for fn in surv_funcs], dtype=float)
    pred_risk = np.clip(pred_risk, 0, 1)

    df = pd.DataFrame({
        "pred_risk": pred_risk,
        "time": np.asarray(y_time),
        "event": np.asarray(y_event).astype(bool)
    }).dropna()

    # quantile bins
    df["bin"] = pd.qcut(df["pred_risk"], q=n_bins, duplicates="drop")

    rows = []
    kmf = KaplanMeierFitter()

    for _, g in df.groupby("bin", observed=False):
        kmf.fit(g["time"], event_observed=g["event"])
        obs_surv = float(kmf.predict(horizon))
        obs_risk = 1 - obs_surv

        rows.append({
            "mean_pred_risk": g["pred_risk"].mean(),
            "obs_risk": obs_risk,
            "n": len(g)
        })

    cal = pd.DataFrame(rows).sort_values("mean_pred_risk").reset_index(drop=True)
    return cal


# example usage
surv_funcs_test = rsf.predict_survival_function(X_test)
t_test = y_test_surv["GTIME"]
y_test_event = y_test_surv["GSTATUS"]

cal_1y = calibration_at_horizon(
    surv_funcs=surv_funcs_test,
    y_time=t_test,
    y_event=y_test_event,
    horizon=365,
    n_bins=7
)

cal_2y = calibration_at_horizon(
    surv_funcs=surv_funcs_test,
    y_time=t_test,
    y_event=y_test_event,
    horizon=730,
    n_bins=7
)

print("1-year calibration table")
print(cal_1y)

print("\n2-year calibration table")
print(cal_2y)

# plot 1-year
plt.figure(figsize=(6, 6))
lim1 = max(cal_1y["mean_pred_risk"].max(), cal_1y["obs_risk"].max()) + 0.01
plt.plot([0, lim1], [0, lim1], "--", color="black")
plt.scatter(cal_1y["mean_pred_risk"], cal_1y["obs_risk"], s=cal_1y["n"]*0.3)
plt.xlim(0, lim1)
plt.ylim(0, lim1)
plt.xlabel("Predicted 1-year risk")
plt.ylabel("Observed 1-year risk")
plt.title("Calibration plot at 1 year")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# plot 2-year
plt.figure(figsize=(6, 6))
lim2 = max(cal_2y["mean_pred_risk"].max(), cal_2y["obs_risk"].max()) + 0.01
plt.plot([0, lim2], [0, lim2], "--", color="black")
plt.scatter(cal_2y["mean_pred_risk"], cal_2y["obs_risk"], s=cal_2y["n"]*0.3)
plt.xlim(0, lim2)
plt.ylim(0, lim2)
plt.xlabel("Predicted 2-year risk")
plt.ylabel("Observed 2-year risk")
plt.title("Calibration plot at 2 years")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
cal_5y = calibration_at_horizon(
    surv_funcs=surv_funcs_test,
    y_time=t_test,
    y_event=y_test_event,
    horizon=365*5,
    n_bins=7
)

print("5-year calibration table")
print(cal_5y)

# plot 5-year
plt.figure(figsize=(6, 6))
lim5 = max(cal_5y["mean_pred_risk"].max(), cal_5y["obs_risk"].max()) + 0.01
plt.plot([0, lim5], [0, lim5], "--", color="black")
plt.scatter(cal_5y["mean_pred_risk"], cal_5y["obs_risk"], s=cal_5y["n"]*0.3)
plt.xlim(0, lim5)
plt.ylim(0, lim5)
plt.xlabel("Predicted 5-year risk")
plt.ylabel("Observed 5-year risk")
plt.title("Calibration plot at 5 years")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ===== Combined calibration figure (A and B) =====

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# -------- Panel A: 1-year --------
ax = axes[0]
ax.text(-0.15, 1.05, "A", transform=ax.transAxes,
        fontsize=14, fontweight="bold")
lim1 = max(cal_1y["mean_pred_risk"].max(), cal_1y["obs_risk"].max()) + 0.01

ax.plot([0, lim1], [0, lim1], "--", color="black")
ax.scatter(
    cal_1y["mean_pred_risk"],
    cal_1y["obs_risk"],
    s=cal_1y["n"] * 0.3
)

ax.set_xlim(0, lim1)
ax.set_ylim(0, lim1)
ax.set_xlabel("Predicted risk")
ax.set_ylabel("Observed risk")
ax.set_title("1-year")
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(plt.MaxNLocator(4, prune='lower'))
ax.yaxis.set_major_locator(plt.MaxNLocator(4, prune='lower'))

# -------- Panel B: 2-year --------
ax = axes[1]
ax.text(-0.15, 1.05, "B", transform=ax.transAxes,
        fontsize=14, fontweight="bold")
lim2 = max(cal_2y["mean_pred_risk"].max(), cal_2y["obs_risk"].max()) + 0.01

ax.plot([0, lim2], [0, lim2], "--", color="black")
ax.scatter(
    cal_2y["mean_pred_risk"],
    cal_2y["obs_risk"],
    s=cal_2y["n"] * 0.3
)

ax.set_xlim(0, lim2)
ax.set_ylim(0, lim2)
ax.set_xlabel("Predicted risk")
ax.set_ylabel("Observed risk")
ax.set_title("2-year")
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(plt.MaxNLocator(4, prune='lower'))
ax.yaxis.set_major_locator(plt.MaxNLocator(4, prune='lower'))

plt.tight_layout()
fig.savefig("calib_cruve.svg", dpi=1200)
fig.savefig("calib_cruve.png", dpi=600)
fig.savefig("calib_cruve.jpg", dpi=1200)
plt.show()

In [ ]:
# ===== Combined calibration figure (A, B, C) =====

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# -------- Panel A: 1-year --------
ax = axes[0]
ax.text(-0.15, 1.05, "A", transform=ax.transAxes,
        fontsize=14, fontweight="bold")

lim1 = max(cal_1y["mean_pred_risk"].max(), cal_1y["obs_risk"].max()) + 0.01

ax.plot([0, lim1], [0, lim1], "--", color="black")
ax.scatter(
    cal_1y["mean_pred_risk"],
    cal_1y["obs_risk"],
    s=cal_1y["n"] * 0.3
)

ax.set_xlim(0, lim1)
ax.set_ylim(0, lim1)
ax.set_xlabel("Predicted risk")
ax.set_ylabel("Observed risk")
ax.set_title("1-year")
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(plt.MaxNLocator(4, prune='lower'))
ax.yaxis.set_major_locator(plt.MaxNLocator(4, prune='lower'))


# -------- Panel B: 2-year --------
ax = axes[1]
ax.text(-0.15, 1.05, "B", transform=ax.transAxes,
        fontsize=14, fontweight="bold")

lim2 = max(cal_2y["mean_pred_risk"].max(), cal_2y["obs_risk"].max()) + 0.01

ax.plot([0, lim2], [0, lim2], "--", color="black")
ax.scatter(
    cal_2y["mean_pred_risk"],
    cal_2y["obs_risk"],
    s=cal_2y["n"] * 0.3
)

ax.set_xlim(0, lim2)
ax.set_ylim(0, lim2)
ax.set_xlabel("Predicted risk")
ax.set_ylabel("Observed risk")
ax.set_title("2-year")
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(plt.MaxNLocator(4, prune='lower'))
ax.yaxis.set_major_locator(plt.MaxNLocator(4, prune='lower'))


# -------- Panel C: 5-year --------
ax = axes[2]
ax.text(-0.15, 1.05, "C", transform=ax.transAxes,
        fontsize=14, fontweight="bold")

lim5 = max(cal_5y["mean_pred_risk"].max(), cal_5y["obs_risk"].max()) + 0.01

ax.plot([0, lim5], [0, lim5], "--", color="black")
ax.scatter(
    cal_5y["mean_pred_risk"],
    cal_5y["obs_risk"],
    s=cal_5y["n"] * 0.3
)

ax.set_xlim(0, lim5)
ax.set_ylim(0, lim5)
ax.set_xlabel("Predicted risk")
ax.set_ylabel("Observed risk")
ax.set_title("5-year")
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_locator(plt.MaxNLocator(4, prune='lower'))
ax.yaxis.set_major_locator(plt.MaxNLocator(4, prune='lower'))


plt.tight_layout()

fig.savefig("calibration_curves.svg", dpi=1200)
fig.savefig("calibration_curves.png", dpi=600)
fig.savefig("calibration_curves.jpg", dpi=1200)

plt.show()

In [ ]:
# reverse event indicator (censoring becomes event)
y_surv_surv = pd.concat([y_train, y_test], axis=0)
reverse_event = ~y_surv_surv["GSTATUS"]

kmf = KaplanMeierFitter()
kmf.fit(y_surv_surv["GTIME"], event_observed=reverse_event)

median_followup = kmf.median_survival_time_
print("Median follow-up:", median_followup)

In [ ]:
from sksurv.metrics import integrated_brier_score, brier_score
import numpy as np

# Define the time points at which to compute the Brier score
surv_funcs = rsf.predict_survival_function(X_test)

times_2 = np.arange(365, 365*5+1, 365)

# Convert survival functions into probabilities at specific time points
preds_ibs = np.asarray([[fn(t) for t in times_2] for fn in surv_funcs])


# Compute Brier Score at specific times
brier_scores = brier_score(y_train_surv, y_test_surv, preds_ibs, times_2)
print(f"Brier Scores: {brier_scores}")

# Compute the Integrated Brier Score (IBS) over the specified time points
ibs = integrated_brier_score(y_train_surv, y_test_surv, preds_ibs, times_2)
print(f"Integrated Brier Score (IBS): {ibs}")


In [ ]:
y_test_event = y_test["GSTATUS"]
y_train_event = y_train["GSTATUS"]

t_test = y_test["GTIME"]
t_train = y_train["GTIME"]

In [ ]:
from sklearn.metrics import roc_curve
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
# 1 ── risk scores
risk_train = rsf.predict(X_train)   
risk_test  = rsf.predict(X_test)

# 2 ── derive Youden-J cut-off on TRAIN ONLY
fpr, tpr, thr = roc_curve(y_train_event, risk_train)          
j_stat        = tpr - fpr
cut_off       = thr[np.argmax(j_stat)]
print(f"training-derived cut-off = {cut_off:.3f}")

# 3 ── assign strata
grp_train = np.where(risk_train > cut_off, "high", "low")
grp_test  = np.where(risk_test  > cut_off, "high", "low")

# 4 ── Kaplan–Meier & log-rank on TEST
kmf = KaplanMeierFitter()
for label in ("low", "high"):
    mask = grp_test == label
    kmf.fit(
        durations=t_test[mask],
        event_observed=y_test_event[mask],
        label=f"{label}-risk"
    ).plot(ci_show=True)

p_val = logrank_test(
    t_test[grp_test == "low"], t_test[grp_test == "high"],
    event_observed_A=y_test_event[grp_test == "low"],
    event_observed_B=y_test_event[grp_test == "high"]
).p_value
print(f"log-rank p-value = {p_val:.3g}")  

In [ ]:
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test
import seaborn as sns

# --- Professional styling ---
sns.set_theme(style="whitegrid", context="talk", rc={
    "axes.edgecolor": "0.2",
    "axes.linewidth": 1.0,
    "grid.color": "0.85", "grid.linestyle": "--",
    "xtick.direction": "in", "ytick.direction": "in",
    "xtick.major.size": 5, "ytick.major.size": 5,
    "font.family": "sans-serif",
    "font.size": 14,
    "axes.labelsize": 14,
    "legend.fontsize": 14
})

# Colors for curves
colors = {"low": "#1f77b4", "high": "#d62728"}

# Create figure
fig, ax = plt.subplots(figsize=(8, 6))

# Plot each group
kmf = KaplanMeierFitter()
for label in ("low", "high"):
    mask = grp_test == label
    kmf.fit(t_test[mask], event_observed=y_test_event[mask],
            label=f"{label.capitalize()} Risk")
    kmf.plot_survival_function(
        ax=ax,
        ci_show=True,
        color=colors[label],
        lw=2.5,
        linestyle="-"
    )

# store fitted KM models
km_models = {}

for label in ("low", "high"):
    mask = grp_test == label
    kmf = KaplanMeierFitter()
    kmf.fit(
        t_test[mask],
        event_observed=y_test_event[mask],
        label=f"{label.capitalize()} Risk"
    )


    km_models[label] = kmf

# Compute and annotate log-rank p-value
p_val = logrank_test(
    t_test[grp_test == "low"], t_test[grp_test == "high"],
    event_observed_A=y_test_event[grp_test == "low"],
    event_observed_B=y_test_event[grp_test == "high"]
).p_value
ax.text(
    0.95, 0.05, r"Log-rank test, $p = {:.3g}$".format(p_val),
    ha="right", va="bottom", transform=ax.transAxes,
    fontsize=11,
    bbox=dict(boxstyle="round,pad=0.3",
              edgecolor="gray", facecolor="white", alpha=0.8)
)

# Labels, grid & spines
ax.set_xlabel("Time (days)")
ax.set_ylabel("Survival Probability")
ax.grid(True, which="both", linestyle="--", alpha=0.4)
sns.despine(trim=True)

# Legend
ax.legend(loc="upper right", frameon=True)

add_at_risk_counts(
    kmf_low, kmf_high,
    ax=ax,
    rows_to_show=['At risk'],
    xticks=[0, 365, 730, 1095, 1460, 1825]   # 0, 1, 2, 3, 4, 5 years
)

# Save and show
fig.tight_layout()
fig.savefig("KM_Survival.svg", dpi=1200)
fig.savefig("KM_Survival.png", dpi=600)
fig.savefig("KM_Survival.jpg", dpi=1200)
plt.show()


# ---- extract 1- and 2-year survival probabilities ----
for label in ("low", "high"):
    surv_1yr = km_models[label].predict(365)
    surv_2yr = km_models[label].predict(730)
    surv_5yr = km_models[label].predict(365*5)

    print(f"{label.capitalize()} risk:")
    print(f"  1-year survival = {surv_1yr:.3f} ({surv_1yr*100:.1f}%)")
    print(f"  2-year survival = {surv_2yr:.3f} ({surv_2yr*100:.1f}%)")
    print(f"  5-year survival = {surv_5yr:.3f} ({surv_5yr*100:.1f}%)")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# --- Style Setup ---
sns.set_theme(style="whitegrid", context="talk", rc={
    "axes.edgecolor": "0.2", "axes.linewidth": 1,
    "xtick.direction": "in", "ytick.direction": "in",
    "grid.color": "0.85", "grid.linestyle": "--",
    "legend.frameon": False, "font.family": "sans-serif",
    "font.size": 14
})

# Your colors
low_color = "#1f77b4"
high_color = "#d62728"

# Data
risk_vec = risk_test
cut_off = cut_off
low = risk_vec[risk_vec <= cut_off]
high = risk_vec[risk_vec > cut_off]

# Bins
bins = np.linspace(0, 800, 500)

# --- Plotting ---
fig, ax = plt.subplots(figsize=(8, 4.5))

# Histograms
ax.hist(low, bins=bins, color=low_color, alpha=0.85,
        label="Low Risk", edgecolor="white", linewidth=0.5)
ax.hist(high, bins=bins, color=high_color, alpha=0.85,
        label="High Risk", edgecolor="white", linewidth=0.5)

# Cut-off line
ax.axvline(cut_off, linestyle="--", linewidth=2, color="k",
           label=f"Cut-off = {cut_off:.2f}")

# X-axis limits—extend 2% beyond data
x_min, x_max = risk_vec.min(), risk_vec.max()
x_pad = 0.02 * (x_max - x_min)
ax.set_xlim(x_min - x_pad, x_max + x_pad)

# Y-axis starts from zero — eliminates blank space
ax.set_ylim(0, None)

# Labels & legend
ax.set_xlabel("Risk Score", fontsize=16)
ax.set_ylabel("Number of Patients", fontsize=16)
ax.legend(loc="upper right", fontsize=14, frameon=True)

sns.despine(offset={'left': -0.7},trim=False)
fig.tight_layout()

# Save

plt.show()


In [ ]:
from dcurves import dca, plot_graphs

horizon = 365
risk_prob = 1.0 - np.array([fn(horizon) for fn in rsf.predict_survival_function(X_test)])

test_dca = pd.DataFrame(y_test)
test_dca["risk"] = risk_prob

In [ ]:
dca_out = dca(data=test_dca, outcome="GSTATUS", thresholds = np.linspace(0, 0.25, 26), time=365, time_to_outcome_col="GTIME", 
              modelnames=["risk"])

# plot_graphs(plot_df=dca_out, y_limits=(-0.05, 0.1))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Professional styling
sns.set_theme(style="whitegrid", context="talk", rc={
    "axes.edgecolor": "0.2", "axes.linewidth": 1,
    "xtick.direction": "in", "ytick.direction": "in",
    "grid.color": "0.85", "grid.linestyle": "--",
    "legend.frameon": False, "font.family": "sans-serif",
    "font.size": 14
})


wide_nb  = dca_out.pivot(index="threshold", columns="model", values="net_benefit")
wide_ia  = dca_out.pivot(index="threshold", columns="model", values="net_intervention_avoided")

# Optional: rename your main model to something nicer for the legend
wide_nb  = wide_nb.rename(columns={"risk":"RSF"})
wide_ia  = wide_ia.rename(columns={"risk":"RSF"})

# 2. Plot setup
fig, ax = plt.subplots(figsize=(9, 6))


line_map = {"RSF": "-", 
            "all": "--", 
            "none": ":"}

colour_map = {"RSF": "#1f78b4", 
            "all": "#e31a1c", 
            "none": "#6a3d9a"}
# 3. Plot curves
for col in wide_nb.columns:
    ax.plot(
        wide_nb.index * 100,
        wide_nb[col],
        label=col,
        color=colour_map.get(col),
        ls=line_map.get(col)
    )

# 4. Labels & title
ax.set_xlabel("Threshold probability (%)", fontsize=14)
ax.set_ylabel("Net benefit", fontsize=14)

# 5. Keep y-axis within your exact limits, add minimal x-axis padding
ax.set_ylim(-0.05, 0.10)
x_min, x_max = wide_nb.index.min() * 100, wide_nb.index.max() * 100
x_pad = 0.01 * (x_max - x_min)
ax.set_xlim(x_min - x_pad, x_max + x_pad)

# 6. Aesthetics
ax.legend(frameon=True, fontsize=14)
sns.despine(ax=ax, trim=True)
ax.grid(True, which="major", linestyle="--", alpha=0.4)
ax.spines['left'].set_position(('outward', 1))
# Expand x-axis limits slightly so gridlines reach the new left edge
x_min, x_max = ax.get_xlim()
ax.set_xlim(x_min - 0.04 * (x_max - x_min), x_max) 
plt.tight_layout()
fig.savefig("DCA.svg", dpi=1200)
fig.savefig("DCA.png", dpi=600)
fig.savefig("DCA.jpg", dpi=1200)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Professional styling
sns.set_theme(style="whitegrid", context="talk", rc={
    "axes.edgecolor": "0.2", "axes.linewidth": 1,
    "xtick.direction": "in", "ytick.direction": "in",
    "grid.color": "0.85", "grid.linestyle": "--",
    "legend.frameon": False, "font.family": "sans-serif",
    "font.size": 14
})


wide_nb  = dca_out.pivot(index="threshold", columns="model", values="net_benefit")
wide_ia  = dca_out.pivot(index="threshold", columns="model", values="net_intervention_avoided")

# Optional: rename your main model to something nicer for the legend
wide_nb  = wide_nb.rename(columns={"risk":"RSF"})
wide_ia  = wide_ia.rename(columns={"risk":"RSF"})

# 2. Plot setup
fig, ax = plt.subplots(figsize=(9, 6))


line_map = {"RSF": "-", 
            "all": "--", 
            "none": ":"}

colour_map = {"RSF": "#1f78b4", 
            "all": "#e31a1c", 
            "none": "#6a3d9a"}
# 3. Plot curves
for col in wide_ia.columns:
    ax.plot(
        wide_ia.index * 100,
        wide_ia[col],
        label=col,
        color=colour_map.get(col),
        ls=line_map.get(col)
    )

# 4. Labels & title
ax.set_xlabel("Threshold probability (%)", fontsize=14)
ax.set_ylabel("Interventions avoided", fontsize=14)

# 5. Keep y-axis within your exact limits, add minimal x-axis padding
ax.set_ylim(-0.1, 1.1)
x_min, x_max = wide_nb.index.min() * 100, wide_nb.index.max() * 100
x_pad = 0.01 * (x_max - x_min)
ax.set_xlim(x_min - x_pad, x_max + x_pad)

# 6. Aesthetics
ax.legend(frameon=True, fontsize=14)
sns.despine(ax=ax, trim=True)
ax.grid(True, which="major", linestyle="--", alpha=0.4)
ax.spines['left'].set_position(('outward', 1))
# Expand x-axis limits slightly so gridlines reach the new left edge
x_min, x_max = ax.get_xlim()
ax.set_xlim(x_min - 0.04 * (x_max - x_min), x_max) 
plt.tight_layout()
fig.savefig("DCA_ia.svg", dpi=1200)
fig.savefig("DCA_ia.png", dpi=600)
fig.savefig("DCA_ia.jpg", dpi=1200)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Professional styling
sns.set_theme(style="whitegrid", context="talk", rc={
    "axes.edgecolor": "0.2", "axes.linewidth": 1,
    "xtick.direction": "in", "ytick.direction": "in",
    "grid.color": "0.85", "grid.linestyle": "--",
    "legend.frameon": False, "font.family": "sans-serif",
    "font.size": 14
})

# 2. Prepare data
wide_nb = dca_out.pivot(index="threshold", columns="model", values="net_benefit")
wide_ia = dca_out.pivot(index="threshold", columns="model", values="net_intervention_avoided")

wide_nb = wide_nb.rename(columns={"risk": "RSF"})
wide_ia = wide_ia.rename(columns={"risk": "RSF"})

line_map = {
    "RSF": "-",
    "all": "--",
    "none": ":"
}

colour_map = {
    "RSF": "#1f78b4",
    "all": "#e31a1c",
    "none": "#6a3d9a"
}

# 3. Create one figure with two panels
fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharex=True)

ax1, ax2 = axes

# ---- Panel A: Net benefit ----
for col in wide_nb.columns:
    ax1.plot(
        wide_nb.index * 100,
        wide_nb[col],
        label=col,
        color=colour_map.get(col),
        ls=line_map.get(col),
        linewidth=2
    )

ax1.set_xlabel("Threshold probability (%)", fontsize=14)
ax1.set_ylabel("Net benefit", fontsize=14)
ax1.set_ylim(-0.05, 0.10)

x_min, x_max = wide_nb.index.min() * 100, wide_nb.index.max() * 100
x_pad = 0.01 * (x_max - x_min)
ax1.set_xlim(x_min - x_pad, x_max + x_pad)

sns.despine(ax=ax1, trim=True)
ax1.grid(True, which="major", linestyle="--", alpha=0.4)
ax1.spines["left"].set_position(("outward", 1))
x1_min, x1_max = ax1.get_xlim()
ax1.set_xlim(x1_min - 0.04 * (x1_max - x1_min), x1_max)

# panel label
ax1.text(-0.12, 1.05, "A", transform=ax1.transAxes,
         fontsize=18, fontweight="bold", va="top", ha="left")

# ---- Panel B: Interventions avoided ----
for col in wide_ia.columns:
    ax2.plot(
        wide_ia.index * 100,
        wide_ia[col],
        label=col,
        color=colour_map.get(col),
        ls=line_map.get(col),
        linewidth=2
    )

ax2.set_xlabel("Threshold probability (%)", fontsize=14)
ax2.set_ylabel("Interventions avoided", fontsize=14)
ax2.set_ylim(-0.1, 1.1)

x_min, x_max = wide_ia.index.min() * 100, wide_ia.index.max() * 100
x_pad = 0.01 * (x_max - x_min)
ax2.set_xlim(x_min - x_pad, x_max + x_pad)

sns.despine(ax=ax2, trim=True)
ax2.grid(True, which="major", linestyle="--", alpha=0.4)
ax2.spines["left"].set_position(("outward", 1))
x2_min, x2_max = ax2.get_xlim()
ax2.set_xlim(x2_min - 0.04 * (x2_max - x2_min), x2_max)

# panel label
ax2.text(-0.12, 1.05, "B", transform=ax2.transAxes,
         fontsize=18, fontweight="bold", va="top", ha="left")

# 4. One shared legend for the whole figure
handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=3, frameon=True, fontsize=13)

# 5. Layout and save
plt.tight_layout(rect=[0, 0, 1, 0.92])

fig.savefig("DCA_combined_AB.svg", dpi=1200, bbox_inches="tight")
fig.savefig("DCA_combined_AB.png", dpi=600, bbox_inches="tight")
fig.savefig("DCA_combined_AB.jpg", dpi=1200, bbox_inches="tight")

plt.show()

In [ ]:
from sklearn.inspection import permutation_importance
perm_imp = permutation_importance(rsf, X_test, y_test_surv, n_repeats=100)

In [ ]:
perm_mean = pd.DataFrame(perm_imp["importances_mean"])
perm_std = pd.DataFrame(perm_imp["importances_std"])
perm_cols = pd.DataFrame(X_test.columns)

In [ ]:
perm_df = pd.concat([perm_cols, perm_mean, perm_std], axis=1)
perm_df.columns = ["var", "mean", "std"]

In [ ]:
perm_df.to_csv("permutation_values.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt

# Sort by mean VIMP for better visualization
df = perm_df.sort_values(by="mean", ascending=False).head(15)

# Plot with error bars
plt.figure(figsize=(10, 6))
plt.barh(df["var"], df["mean"], xerr=df["std"], capsize=5, color="skyblue", edgecolor="black")
plt.xlabel("Mean VIMP")
plt.ylabel("Variables")
plt.title("Variable Importance with Standard Deviation")
plt.gca().invert_yaxis()  # Highest importance at the top
plt.show()


In [ ]:
#same as the above file just modified the varaibel names beunderstandable in the figure
perm_df = pd.read_csv("permutation_values2.csv")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# style setup
sns.set_theme(style="whitegrid", context="talk", rc={
    "axes.edgecolor": "0.2",
    "axes.linewidth": 1,
    "grid.color": "0.85",
    "grid.linestyle": "--",
    "xtick.direction": "in",
    "ytick.direction": "in",
    "font.family": "sans-serif",
    "font.size": 12
})

# sort and keep top 15
df = perm_df.sort_values("mean", ascending=False).head(15).copy()
df = df[::-1]  # invert for plotting top at top

# plot
fig, ax = plt.subplots(figsize=(10, 8))

ax.tick_params(axis='y', labelsize=12)  # adjust to your desired font size

bars = ax.barh(
    df["var"],
    df["mean"],
    xerr=df["std"],
    color="#1f78b4",
    edgecolor="black",
    capsize=4,
    linewidth=1
)

# # add mean value text to each bar
# for bar in bars:
#     w = bar.get_width()
#     ax.text(w + 0.005 * (df["mean"].max()),
#             bar.get_y() + bar.get_height() / 2,
#             f"{w:.3f}",
#             va="center",
#             ha="left",
#             fontsize=11)

ax.set_xlabel("Mean VIMP", fontsize=16)
ax.set_ylabel("Variable", fontsize=16)
ax.set_xlim(df["mean"].min()-df["std"].max() * 1.1, df["mean"].max() + df["std"].max() * 1.1)  # ensures error bars aren't clipped

# sns.despine(left=False, bottom=False)
ax.grid(axis="x", linestyle="--", alpha=0.4)

fig.tight_layout()
fig.savefig("VIMP_Fig.svg", dpi=1200)
fig.savefig("VIMP_Fig.png", dpi=600)
fig.savefig("VIMP_Fig.jpg", dpi=1200)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

def partial_dependence_at_horizon(model, X, feature, grid, horizon=365):
    """
    For each value in grid, set X[feature] = value for all rows,
    compute predicted survival at horizon, convert to risk (1 - S),
    and return the mean across patients.
    """
    X_copy = X.copy()
    mean_risks = []
    for v in grid:
        X_copy[feature] = v
        surv_funcs = model.predict_survival_function(X_copy)
        risks = 1 - np.array([fn(horizon) for fn in surv_funcs])
        mean_risks.append(risks.mean())
    return np.array(mean_risks)

ischemia_col = "ISCHTIME"

# Use observed range in the test set, clipped to trim extreme tails
low, high = np.quantile(X_test[ischemia_col], [0.02, 0.98])
grid = np.linspace(low, high, 30)

pd_1y = partial_dependence_at_horizon(rsf, X_test, ischemia_col, grid, horizon=365)
pd_2y = partial_dependence_at_horizon(rsf, X_test, ischemia_col, grid, horizon=730)


# --- Plot ---
sns.set_theme(style="whitegrid", context="talk", rc={
    "axes.edgecolor": "0.2", "axes.linewidth": 1,
    "xtick.direction": "in", "ytick.direction": "in",
    "grid.color": "0.85", "grid.linestyle": "--",
    "font.family": "sans-serif", "font.size": 14
})

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(grid, pd_1y, "o-", color="#1f78b4", lw=2.2, label="1-year risk")
ax.plot(grid, pd_2y, "s-", color="#d62728", lw=2.2, label="2-year risk")



ax.set_xlim(0, 6.5)
ax.set_ylim(0.04, 0.15)

ax.set_xlabel("Ischemia time (hours)")
ax.set_ylabel("Average predicted risk of graft failure")
ax.set_title("Partial dependence: ischemia time")
ax.legend(loc="upper left", frameon=True)
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig("pdp_ischemia.svg", dpi=1200)
fig.savefig("pdp_ischemia.png", dpi=600)
fig.savefig("pdp_ischemia.jpg", dpi=1200)
plt.show()

print("\nIschemia time (hours) → avg predicted risk")
print(f"{'hours':>7} {'1-year':>10} {'2-year':>10}")
for g, r1, r2 in zip(grid, pd_1y, pd_2y):
    print(f"{g:7.2f} {r1:10.4f} {r2:10.4f}")
